# **Single-Cell Analysis**

This notebook processes the filtered single-nucleus dataset (`adata_filtered.h5ad`) prepared in `reduction_of_dataset.ipynb`.

It performs normalization, log-transformation, HVG selection, PCA, UMAP, Leiden clustering and marker identification.

The processed object (`adata_pp.h5ad`) will be used for cluster annotation and pseudobulk differential expression (in `translation_to_R.ipynb`).

---

**Pipeline Overview:**
- Setup and Configuration
- Data Preparation (Normalization, HVG, Scaling)
- Dimensionality Reduction (PCA, UMAP, Leiden)
- Quality Checks
- Marker Gene Identification
- Save Processed Object

**Note:** This pipeline does NOT perform batch correction or integration. The AD/PD/CTRL cohort is too small for reliable integration, particularly for PD (only 3 donors).

# **Setup block**


### Setup and Configuration

In [1]:
# ==========================================================================================================
# GESTION SYSTÈME & ENVIRONNEMENT
import os                                       # Navigation fichiers (DIRS, chemins relatifs)
import gc                                       # Gestion mémoire (nettoyage objets inutilisés)
import json                                     # Lecture du dictionnaire de métadonnées (pipeline / EDA)  
import warnings                                 # Masquer warnings Scanpy/AnnData (dépréciation)
import re                                       # Parsing noms échantillons (regex pour donor_id)
from IPython.display import Markdown, display   # Affichage Jupyter (titres formatés, tableaux HTML)


# ----------------------------------------------------------------------------------------------------------
# CALCUL NUMÉRIQUE & VISUALISATION
# >> Générer les QC plots (n_genes vs n_counts, % mitochondrial, distributions par pathologie)
import numpy as np                              # Matrices creuses (X sparse), seuils QC (percentiles)
import math                                     # Calculs grilles subplots (ceil/floor pour layout)
import matplotlib.pyplot as plt                 # Figures principales (violin, scatter, hist)
import seaborn as sns                           # Heatmaps corrélations, boxplots stylisés

# ----------------------------------------------------------------------------------------------------------
# ANALYSE SINGLE-CELL
# >> Charger données brutes (ad.read_h5ad), appliquer filtres QC (mitochondrial %, doublets),
#    calculer métriques (sc.pp.calculate_qc_metrics)
import anndata as ad                            # Objet AnnData (adata.obs, adata.var, adata.X)
import scanpy as sc                             # Fonctions QC (sc.pp.filter_cells, sc.pl.violin)
import pandas as pd                             # Métadonnées (fusion adata.obs), tableaux récapitulatifs

# ----------------------------------------------------------------------------------------------------------
# CLUSTERING (PRÉPARATION POUR S2 - NON UTILISÉ DANS S1)
# >> Importé par anticipation pour pipeline complet (normalisation → Leiden)
import leidenalg                                # Algorithme Leiden (clustering post-normalisation)
import igraph                                   # Graphe k-NN (requis pour leidenalg sous Scanpy)

# ==========================================================================================================
# --- DÉFINITION DES DOSSIERS ---
PROJECT_ROOT = "C:\\Z\\M2_AIDA\\transcriptomics_project"  # Laïla

DIRS = {
    "DATA":    os.path.join(PROJECT_ROOT, "data"),
    "EDA":     os.path.join(PROJECT_ROOT, "eda"),
    "FIGURES": os.path.join(PROJECT_ROOT, "figures"),
    "TMP":     os.path.join(PROJECT_ROOT, "tmp_cache")
}

for path in DIRS.values():
    os.makedirs(path, exist_ok=True)

os.chdir(PROJECT_ROOT)

# ==========================================================================================================
# --- PARAMÈTRES SCANPY ---
sc.settings.figdir = DIRS["FIGURES"]
sc.settings.cachedir = DIRS["TMP"]
sc.settings.datasetdir = DIRS["DATA"]
sc.settings.set_figure_params(dpi=100, frameon=False)

warnings.filterwarnings("ignore") 

# ==========================================================================================================
print(f"✅ Environnement chargé. Working directory: {os.getcwd()}")

✅ Environnement chargé. Working directory: c:\Z\M2_AIDA\transcriptomics_project


# **Data Loading**

### Dataset analytique Post-QC :

<br>  local:       `data/HBCC_SCZ_CTRL_postQC.h5ad`.

In [8]:
dataset_path = os.path.join(DIRS["DATA"], "HBCC_SCZ_CTRL_postQC.h5ad")
# adata = sc.read_h5ad(dataset_path) >>>>> MemoryError !
adata = sc.read_h5ad(dataset_path, backed='r')  # Lecture en mode backed pour économiser la mémoire
# ** Note technique :** backed mode pour éviter MemoryError.

dictionary_path = os.path.join(DIRS["DATA"],"hbcc_cellxgene_metadata_dictionary.json")
with open(dictionary_path, "r") as f:
    meta_hbcc = json.load(f)

print(f"✅ HBCC_SCZ_CTRL_postQC and metadata dictionary successfully loaded (backed mode). Shape (cells, genes): {adata.shape}")

✅ HBCC_SCZ_CTRL_postQC and metadata dictionary successfully loaded (backed mode). Shape (cells, genes): (414669, 34176)


## XXX

#### **Single-Cell Structure and Cell-Type Representation**

<small>
Cette étape vise à caractériser la structure cellulaire du cortex préfrontal humain dans la cohorte post-QC schizophrénie / contrôles. L’objectif est d’explorer la composition en types cellulaires, de vérifier la cohérence des annotations biologiques et d’évaluer la représentation relative des différents types cellulaires entre les groupes diagnostiques.  
<br><br>
Cette analyse est purement descriptive et exploratoire : aucun reclustering ni analyse différentielle n’est réalisé à ce stade. Elle constitue une étape de validation biologique indispensable avant d’engager des analyses transcriptomiques plus approfondies.  
<br><br>
Les annotations de types cellulaires utilisées proviennent des métadonnées associées à l’atlas HBCC et reflètent une classification experte basée sur des signatures transcriptomiques connues du cortex humain.
</small>


In [9]:
celltype_counts = adata.obs["cell_type"].value_counts()

print("Nombre de cellules par type cellulaire (cohorte SCZ / CTRL) :\n")
print(celltype_counts)


Nombre de cellules par type cellulaire (cohorte SCZ / CTRL) :

cell_type
oligodendrocyte                                                   102662
astrocyte                                                          49099
L2/3 intratelencephalic projecting glutamatergic neuron            45914
oligodendrocyte precursor cell                                     35022
L2/3-6 intratelencephalic projecting glutamatergic neuron          33751
VIP GABAergic cortical interneuron                                 22245
sst GABAergic cortical interneuron                                 20794
microglial cell                                                    20282
pvalb GABAergic cortical interneuron                               20254
endothelial cell                                                    9914
GABAergic neuron                                                    9476
lamp5 GABAergic cortical interneuron                                9302
L6 intratelencephalic projecting glutamatergic neur

#### **Interpretation of Global Cell-Type Composition**

<small>
L’analyse de la composition cellulaire globale de la cohorte post-QC schizophrénie / contrôles met en évidence une organisation conforme aux attentes biologiques pour le cortex préfrontal humain. Les populations gliales dominent largement le jeu de données, en particulier les oligodendrocytes et les astrocytes, reflétant leur abondance physiologique et leur rôle central dans le fonctionnement et le maintien du tissu cortical.  
<br><br>
Les neurones glutamatergiques corticaux constituent la principale population neuronale, avec une représentation importante des neurones intratelencephaliques des couches superficielles (L2/3 et L2/3–6), cohérente avec l’architecture laminaire du cortex. Les neurones inhibiteurs GABAergiques sont également bien représentés, incluant plusieurs sous-types majeurs (VIP, SST, PVALB, LAMP5), témoignant de la diversité interne des circuits inhibiteurs corticaux.  
<br><br>
Les populations non neuronales minoritaires, telles que les cellules endothéliales, péricytes, macrophages périvasculaires et cellules immunitaires périphériques, sont présentes en proportions plus faibles, ce qui est attendu dans un contexte de single-nucleus RNA-seq du cortex.  
<br><br>
Dans l’ensemble, cette distribution confirme que la cohorte post-QC conserve une structure cellulaire biologiquement plausible et représentative du cortex préfrontal humain. Elle valide l’utilisation de ce jeu de données pour l’exploration ultérieure des différences entre groupes diagnostiques, ainsi que pour des analyses ciblées par type cellulaire.
</small>


#### **Cell-Type Composition Stratified by Diagnostic Group (SCZ vs CTRL)**

<small>
Après avoir établi la composition cellulaire globale de la cohorte post-QC, cette étape vise à examiner la répartition des types cellulaires en fonction du groupe diagnostique, en comparant les individus atteints de schizophrénie (SCZ) aux contrôles.  
<br><br>
L’objectif est de vérifier si la structure cellulaire globale est conservée entre les groupes diagnostiques et d’identifier d’éventuelles différences de représentation relatives des principaux types cellulaires. Cette analyse est strictement descriptive et ne constitue pas une analyse différentielle d’expression génique.  
<br><br>
L’exploration de la composition cellulaire par diagnostic permet de s’assurer que les comparaisons transcriptomiques ultérieures ne sont pas dominées par des déséquilibres majeurs de proportions cellulaires, et fournit un cadre interprétatif essentiel pour les analyses en aval.
</small>


In [10]:
celltype_by_disease = (
    adata.obs
    .groupby(["cell_type", "disease"])
    .size()
    .unstack(fill_value=0)
)

print("Nombre de cellules par type cellulaire et groupe diagnostique (SCZ vs contrôles) :\n")
print(celltype_by_disease)


Nombre de cellules par type cellulaire et groupe diagnostique (SCZ vs contrôles) :

disease                                             bipolar II disorder  \
cell_type                                                                 
astrocyte                                                          1443   
endothelial cell                                                    187   
GABAergic neuron                                                    108   
oligodendrocyte precursor cell                                      590   
oligodendrocyte                                                    1303   
T cell                                                               11   
natural killer cell                                                  14   
B cell                                                                1   
plasma cell                                                           0   
microglial cell                                                     283   
perivascular mac

#### **Interpretation of Cell-Type Composition Stratified by Diagnostic Group**

<small>
La stratification de la composition cellulaire par groupe diagnostique met en évidence une forte hétérogénéité des catégories associées à la variable <code>disease</code>. Au-delà des groupes attendus (schizophrénie et contrôles), de nombreuses cellules sont annotées avec des diagnostics composites combinant plusieurs troubles psychiatriques, neurologiques ou métaboliques.  
<br><br>
Cette observation indique que la variable <code>disease</code> encode non seulement un diagnostic principal, mais également des comorbidités, sous forme de chaînes de caractères concaténées. En conséquence, les groupes diagnostiques ne correspondent pas encore à des catégories homogènes directement comparables.  
<br><br>
La présence de ces diagnostics composites souligne l’importance d’une étape de simplification et de recodage des groupes diagnostiques avant toute analyse comparative ou différentielle. Cette étape permettra de définir des groupes biologiquement interprétables, en isolant par exemple les individus atteints de schizophrénie sans comorbidités majeures et les contrôles, tout en excluant ou en traitant séparément les cas complexes.  
<br><br>
Ainsi, cette analyse descriptive remplit un rôle clé de validation et met en évidence la nécessité d’un filtrage diagnostique explicite, qui constituera l’étape suivante du pipeline.
</small>


#### **Diagnostic Group Refinement and Cohort Cleaning**

<small>
L’analyse de la composition cellulaire stratifiée par diagnostic a mis en évidence une forte hétérogénéité de la variable <code>disease</code>, qui inclut de nombreux diagnostics composites combinant plusieurs troubles psychiatriques, neurologiques ou somatiques. Cette structure reflète la richesse clinique du jeu de données HBCC, mais ne permet pas une comparaison directe et interprétable entre groupes biologiquement homogènes.  
<br><br>
L’objectif de cette étape est de simplifier et de clarifier la définition des groupes diagnostiques afin de constituer une cohorte adaptée aux analyses comparatives ultérieures. En particulier, l’analyse se concentrera sur la comparaison entre individus atteints de schizophrénie et individus contrôles, en excluant les cas présentant des comorbidités complexes susceptibles de confondre l’interprétation transcriptomique.  
<br><br>
Cette étape consiste donc à recoder les annotations diagnostiques en catégories simplifiées, à filtrer explicitement les groupes d’intérêt et à produire un jeu de données nettoyé sur le plan clinique. Elle constitue une transition essentielle entre l’exploration descriptive de la structure cellulaire et les analyses transcriptomiques ciblées réalisées en aval.
</small>


In [11]:
adata.obs["simple_diagnosis"] = "OTHER"

adata.obs.loc[
    adata.obs["disease"] == "schizophrenia",
    "simple_diagnosis"
] = "SCZ"

adata.obs.loc[
    adata.obs["disease"] == "normal",
    "simple_diagnosis"
] = "CTRL"

mask = adata.obs["simple_diagnosis"].isin(["SCZ", "CTRL"])

adata_clean = adata[mask]

print("Cellules en entrée :", adata.n_obs)
print("Cellules retenues après simplification diagnostique :", adata_clean.n_obs)
print("\nRépartition des groupes diagnostiques :")
print(adata_clean.obs["simple_diagnosis"].value_counts())


Cellules en entrée : 414669
Cellules retenues après simplification diagnostique : 232274

Répartition des groupes diagnostiques :
simple_diagnosis
CTRL    205073
SCZ      27201
Name: count, dtype: int64


#### **Interpretation of Diagnostic Group Refinement**

<small>
La simplification des annotations diagnostiques a permis de définir une cohorte cliniquement homogène adaptée aux analyses comparatives ultérieures. À partir de 414 669 cellules issues de la cohorte post-QC initiale, 232 274 cellules ont été retenues après application d’un filtrage strict basé sur le diagnostic principal.  
<br><br>
Cette cohorte nettoyée comprend deux groupes clairement définis : un groupe contrôle (CTRL) regroupant 205 073 cellules, et un groupe schizophrénie (SCZ) regroupant 27 201 cellules. Les cellules associées à des diagnostics composites ou à des comorbidités complexes ont été exclues afin de limiter les facteurs de confusion biologiques et cliniques.  
<br><br>
La différence d’effectif entre les groupes reflète la structure du jeu de données original et n’est pas inattendue dans un atlas populationnel. Cette cohorte simplifiée constitue une base robuste pour l’exploration des différences transcriptomiques entre schizophrénie et contrôles, notamment dans des analyses stratifiées par type cellulaire ou sous forme de pseudobulk.
</small>


#### **Selection of Major Cortical Cell Types for Downstream Analyses**

<small>
Après la définition d’une cohorte diagnostique homogène schizophrénie / contrôles, l’étape suivante consiste à sélectionner les principaux types cellulaires du cortex préfrontal qui seront considérés pour les analyses transcriptomiques en aval.  
<br><br>
L’objectif de cette sélection est double. D’une part, elle permet de réduire la complexité du jeu de données en se concentrant sur les populations cellulaires les plus abondantes et les plus pertinentes biologiquement dans le contexte de la schizophrénie. D’autre part, elle vise à garantir une puissance analytique suffisante pour chaque type cellulaire retenu, en excluant les populations très rares susceptibles de produire des résultats instables ou difficilement interprétables.  
<br><br>
Cette étape repose sur les annotations de types cellulaires fournies par l’atlas HBCC et s’inscrit dans une logique descriptive et préparatoire. Elle ne constitue pas encore une analyse différentielle, mais prépare la constitution de sous-ensembles cellulaires cohérents qui serviront de base aux analyses transcriptomiques ciblées et aux approches de type pseudobulk.
</small>


In [13]:
adata_clean_mem = adata_clean.to_memory()

major_cell_types = [
    "oligodendrocyte",
    "astrocyte",
    "microglial cell",
    "oligodendrocyte precursor cell",
    "L2/3 intratelencephalic projecting glutamatergic neuron",
    "L2/3-6 intratelencephalic projecting glutamatergic neuron",
    "L6 intratelencephalic projecting glutamatergic neuron",
    "L6b glutamatergic neuron of the primary motor cortex",
    "L5/6 near-projecting glutamatergic neuron",
    "pvalb GABAergic cortical interneuron",
    "sst GABAergic cortical interneuron",
    "VIP GABAergic cortical interneuron",
    "lamp5 GABAergic cortical interneuron"
]

mask_ct = adata_clean_mem.obs["cell_type"].isin(major_cell_types)
adata_ct = adata_clean_mem[mask_ct]

print("Cellules en entrée :", adata_clean_mem.n_obs)
print("Cellules retenues après sélection des types cellulaires :", adata_ct.n_obs)
print("\nRépartition des types cellulaires retenus :")
print(adata_ct.obs["cell_type"].value_counts())


Cellules en entrée : 232274
Cellules retenues après sélection des types cellulaires : 209093

Répartition des types cellulaires retenus :
cell_type
oligodendrocyte                                              58166
astrocyte                                                    26309
L2/3 intratelencephalic projecting glutamatergic neuron      23451
oligodendrocyte precursor cell                               21762
L2/3-6 intratelencephalic projecting glutamatergic neuron    15987
microglial cell                                              13673
sst GABAergic cortical interneuron                           12550
VIP GABAergic cortical interneuron                           12533
pvalb GABAergic cortical interneuron                         11385
lamp5 GABAergic cortical interneuron                          5115
L6 intratelencephalic projecting glutamatergic neuron         3908
L6b glutamatergic neuron of the primary motor cortex          2410
L5/6 near-projecting glutamatergic neuron       

#### **Interpretation of Major Cell-Type Selection**

<small>
La sélection des types cellulaires majeurs a permis de constituer un sous-ensemble biologiquement pertinent du cortex préfrontal, tout en conservant l’essentiel de la diversité cellulaire du jeu de données. À partir de 232 274 cellules appartenant à la cohorte schizophrénie / contrôles, 209 093 cellules ont été retenues après exclusion des populations rares ou peu représentées.  
<br><br>
Les populations conservées correspondent aux principaux types cellulaires attendus dans le cortex humain adulte, incluant les oligodendrocytes et leurs précurseurs, les astrocytes, les microglies, ainsi que plusieurs sous-types neuronaux glutamatergiques et GABAergiques. Les oligodendrocytes et les astrocytes représentent les populations les plus abondantes, reflétant leur rôle central dans l’organisation et le fonctionnement du tissu cortical.  
<br><br>
Les interneurones GABAergiques (pvalb, sst, VIP, lamp5) et les différents sous-types de neurones glutamatergiques intratélencéphaliques sont également bien représentés, garantissant une base suffisante pour des analyses comparatives spécifiques à chaque type cellulaire.  
<br><br>
L’exclusion des types cellulaires rares (notamment certaines populations vasculaires ou immunitaires faiblement représentées) permet de limiter le bruit et d’assurer une puissance statistique adéquate pour les analyses transcriptomiques en aval. Cette sélection constitue ainsi un compromis raisonné entre complexité biologique et robustesse analytique, préparant le jeu de données aux étapes de normalisation, de réduction de dimension et d’analyse différentielle.
</small>


#### **Normalization and Preparation for Downstream Single-Cell Analyses**

<small>
Après la sélection d’une cohorte diagnostique homogène (schizophrénie / contrôles) et des principaux types cellulaires du cortex préfrontal, les données sont désormais prêtes pour les étapes centrales de l’analyse single-cell transcriptomique.  
<br><br>
L’objectif de cette étape est de rendre les profils d’expression comparables entre cellules en corrigeant les différences de profondeur de séquençage, puis de transformer les données afin de stabiliser la variance. Cette normalisation est une condition indispensable à la réduction de dimension, au clustering et aux analyses différentielles en aval.  
<br><br>
Le prétraitement comprend classiquement une normalisation des comptages par cellule, suivie d’une transformation logarithmique. Ces opérations permettent de conserver l’information biologique tout en limitant l’impact des variations techniques inhérentes aux données single-nucleus RNA-seq.  
<br><br>
Cette étape marque la transition entre la phase de préparation des données et le cœur de l’analyse transcriptomique, qui reposera sur l’identification des principales sources de variation et la structuration de l’espace d’expression génique.
</small>
